# Colab: fine-tuning groups (now: `label_eff_extra`, 2 more seeds for label efficiency)

Runs the same `finetune.py` jobs as Kaggle Notebook 02, on Colab's free T4. Same splits, same encoder, same code. Progress is saved to **Google Drive after every fold**, so if Colab disconnects, just run all cells again and it continues.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU.**
2. **Kaggle API token:** on kaggle.com → your profile → **Settings → API → Create New Token**. This downloads `kaggle.json`; cell 1 asks you to upload it the first time and then keeps it in your Drive.
3. **Keep this browser tab open.** The free tier disconnects idle sessions after roughly 90 minutes without interaction, and every session ends after at most 12 h. If that happens, **Runtime → Run all** again.

**Scripts:** if the file `jbhi_experiments.zip` is in your Drive folder (`My Drive/jbhi/`), the notebook uses it; otherwise it downloads the Kaggle dataset `jbhi-experiments`. Cell 3 stops if the scripts are outdated.

**At the end:** cell 5 makes `colab_runs.zip` in your Drive. Upload it to Kaggle as a new private dataset (e.g. `colab-runs`) and attach it to Notebook 02. Cell 1 there merges the runs automatically.

## 0 — Settings (Kaggle dataset names)

In [ ]:
DATASETS = {
    'scripts': 'mirmashrafi/jbhi-experiments-8',                     # your scripts (latest version)
    'splits':  'mirmashrafi/setup-1-output',                       # splits/windows.csv
    'memmap':  'phantom8/downtream-hybrid-ckpts-ppg-abp-ecg',      # physio_memmap
    'weights': 'phantom8/pretrain-v6-weights',                     # thesis encoder
}
RUN_GROUPS = ['label_eff_extra']
DRIVE_DIR = '/content/drive/MyDrive/jbhi'                          # results + kaggle.json live here

## 1 — Google Drive and Kaggle API token

In [ ]:
import os, json, shutil, glob, time
from google.colab import drive, files
drive.mount('/content/drive')
os.makedirs(DRIVE_DIR, exist_ok=True)
tok = f'{DRIVE_DIR}/kaggle.json'
if not os.path.exists(tok):
    print('Upload kaggle.json (Kaggle -> Settings -> API -> Create New Token):')
    up = files.upload()
    open(tok, 'wb').write(up['kaggle.json'])
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy(tok, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2 — Download the data (a few minutes; skipped if already there)

In [ ]:
DATA = '/content/data'
for key, slug in DATASETS.items():
    dst = f'{DATA}/{key}'
    if key == 'scripts':
        shutil.rmtree(dst, ignore_errors=True)          # scripts are small: always fetch the newest
    if os.path.isdir(dst) and os.listdir(dst):
        print(f'[have] {key}')
        continue
    os.makedirs(dst, exist_ok=True)
    print(f'[download] {slug}')
    !kaggle datasets download -d {slug} -p {dst} --unzip -q
    print('   ', sorted(os.listdir(dst))[:10])

## 3 — Find everything (same logic as Kaggle Notebook 02)

In [ ]:
def find(pattern):
    return sorted(glob.glob(pattern, recursive=True))

# scripts: a jbhi_experiments.zip in your Drive folder wins over the Kaggle dataset
import zipfile
dz = f'{DRIVE_DIR}/jbhi_experiments.zip'
if os.path.exists(dz):
    shutil.rmtree(f'{DATA}/scripts_drive', ignore_errors=True)
    zipfile.ZipFile(dz).extractall(f'{DATA}/scripts_drive')
    s = find(f'{DATA}/scripts_drive/**/finetune.py')
    print('scripts from Drive:', dz)
else:
    s = find(f'{DATA}/scripts/**/finetune.py')
    print('scripts from Kaggle:', DATASETS['scripts'])
assert s, 'scripts not found -- check DATASETS["scripts"]'
shutil.rmtree('/content/exp', ignore_errors=True)
shutil.copytree(os.path.dirname(s[0]), '/content/exp')
os.chdir('/content/exp')
src = open('finetune.py').read()
assert "'--mask_start'" in src and os.path.exists('predict_thesis.py'), (
    'These scripts are an OLD version. Put the newest jbhi_experiments.zip into ' + DRIVE_DIR +
    ' (or upload it as a new version of the Kaggle dataset) and run all again.')
print('scripts version OK:', open('VERSION').read().strip() if os.path.exists('VERSION') else '?')

mm = find(f'{DATA}/memmap/**/train/offset_5m/n_samples.npy')
assert mm, 'memmaps not found -- check DATASETS["memmap"]'
MEM = os.path.dirname(os.path.dirname(os.path.dirname(mm[0])))
spl = find(f'{DATA}/splits/**/splits/windows.csv')
assert spl, 'windows.csv not found -- check DATASETS["splits"]'
SPL = spl[0]
enc = find(f'{DATA}/weights/**/hybrid_v4_encoders.pth')
assert enc, 'hybrid_v4_encoders.pth not found -- check DATASETS["weights"]'
ENC = enc[0]

RUNS = f'{DRIVE_DIR}/runs'                     # on Drive: survives disconnects, enables --resume
os.makedirs(RUNS, exist_ok=True)
print('memmaps :', MEM)
print('splits  :', SPL)
print('encoder :', ENC)
print('runs    :', RUNS, sorted(os.listdir(RUNS)))

## 4 — Run the jobs (one GPU, one job at a time)

In [ ]:
T0 = time.time()
DEADLINE = T0 + 11 * 3600
COMMON = f'--memmap_dir "{MEM}" --windows_csv "{SPL}" --out_dir "{RUNS}" --resume --deadline {DEADLINE} --num_workers 2'
P = f'--init pretrained --encoder_path "{ENC}"'
W = '--windows offset_5m offset_15m'
GROUPS = {
    'frozen': {'frozen': f'{P} --freeze_encoder {W}'},
    'modalities_short': {**{f'single_{m}': f'{P} --modalities {m} {W}' for m in ['ppg', 'abp', 'ecg']},
                         **{f'drop_{x}': f'{P} --modalities {",".join(m for m in ["ppg", "abp", "ecg"] if m != x)} {W}'
                            for x in ['ppg', 'abp', 'ecg']}},
    # seeds 3 and 4 for the scarce-label fractions (seeds 0-2 ran on Kaggle in group label_eff)
    'label_eff_extra': {f'le_{v}_f{f}_s{sd}': (P if v == 'ssl' else '--init scratch') +
                        f' --windows offset_5m --label_frac {f} --seed {sd}'
                        for f in [0.1, 0.25] for sd in [3, 4] for v in ['ssl', 'scratch']},
}
done = lambda n: os.path.exists(f'{RUNS}/{n}/summary.csv')
for g in RUN_GROUPS:
    for name, flags in GROUPS[g].items():
        if done(name):
            print(f'[done] {name}')
            continue
        if time.time() > DEADLINE:
            print(f'[deadline] not starting {name}')
            continue
        print(f'\n########## {name}  ({(time.time() - T0) / 3600:.1f} h) ##########', flush=True)
        !python -u finetune.py {COMMON} {flags} --run_name {name} 2>&1 | grep --line-buffered -v "UserWarning\|self.transformer = nn"
print('\nfinished:', [n for g in RUN_GROUPS for n in GROUPS[g] if done(n)])
print('not yet :', [n for g in RUN_GROUPS for n in GROUPS[g] if not done(n)])

## 5 — Pack the results for Kaggle

Makes `colab_runs_<group>.zip` (folder `runs/` inside) in your Drive, with this commit's groups only. Upload it to Kaggle: **Create → New Dataset**, private, e.g. `colab-runs`; Kaggle unzips it. Then attach that dataset in Notebook 02 together with the other inputs.

Only finished runs are packed. Unfinished ones stay in Drive and continue when you run this notebook again.

In [ ]:
import zipfile
out = f'{DRIVE_DIR}/colab_runs_{"_".join(RUN_GROUPS)}.zip'
mine = {n for g in RUN_GROUPS for n in GROUPS[g]}          # only this notebook's current groups
n = 0
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for d in sorted(os.listdir(RUNS)):
        if d not in mine or not done(d):
            continue
        for f in glob.glob(f'{RUNS}/{d}/*'):
            if os.path.isfile(f) and not f.endswith('.pth'):          # predictions + summaries only
                z.write(f, f'runs/{d}/{os.path.basename(f)}')
                n += 1
print(f'{out}: {n} files from', sorted(d for d in mine if done(d)))